# BPE
Author : Mahanth Yalla 

##  Imports

In [1]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
# import seaborn as sns
from tqdm import tqdm


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
data_dir = 'data/'
models_dir = 'models/'

for a small language model, we need reasonable tokenizer, so wee need a custom tokenizer 

let build a Tokenizer using 
## Byte-Pair Encoding algorithm


### Version 1 : Greedy

Few issuses known in Tokenization
* Tokenizing effects the indentation of progamming lang like python -> out of context len soon
* same word such as `dog,` `dog.` `Dog` `dog!` are mapped to different tokens [GPT-2 Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
* Arthemetic is not possible as tokenization splits a number into $k$ tokens 
* larger the encoding -> better the distinction between words and will be robust -> use `UNICODE` : ~150K chars defined
* **BEST UTF** : UTF-8 -> 1 to 4 B (backward compatible to ascii)
* 


In [2]:
text = "Basic LatinABCD abcd 0123 ?#$%Latin Extended A-BĀĂĄ ĆĈĊ ĒĔĖĘ Ⱡ Ɽ Ɱ ⱲLatin IPA Extentionsɖɜɣ ɘɫɛ ɱɷɞSpacing Modifierspʰ pʱ pʲ pʳDiacritical Marksàáâã èéêẽ òóôõGreek and CopticΑ Β Γ Δ Ε Ζ Η Θ Ι Κ ΛSlavic / CyrillicА Б В Г Д Е Ё Ж З И ЙGeorgianႠ Ⴁ Ⴂ Ⴃ Ⴄ Ⴅ Ⴆ Ⴇ Ⴈ Ⴉ ႪArmenianԱ Բ Գ Դ Ե Զ Է Ը Թ ԺGlagoliticⰀ Ⰱ Ⰲ Ⰳ Ⰴ Ⰵ Ⰶ Ⰷ Ⰸ ⰉArabicا ب ت ث ج ح خ د ذ ر زHebrewב ג ד ה ו ז ח ט י כ לSyriacܐ ܒ ܓ ܕ ܗ ܘ ܙ ܚ ܛ ܝ ܟHindi / Devanagariक ख ग घ ङ च छ ज झ ञ टThaiก ข ฃ ค ฅ ฆ ง จ ฉ ช ซEthiopicሀ ሁ ሂ ሃ ሄ ህ ሆ ሇ ለ ሉ ሊCherokeeᎠ Ꭱ Ꭲ Ꭳ Ꭴ Ꭵ Ꭶ Ꭷ Ꭸ Ꭹ ᎪAboriginalᐄ ᐅ ᐆ ᐇ ᐈ ᐉ ᐊ ᐋ ᐌ ᐍ ᐎRunicᚠ ᚢ ᚦ ᚨ ᚱ ᚲ ᚷ ᚹ ᚺ ᚾ ᛁGeneral Punctuation‰ ‱ ⁒ ‼ ⁇ ⁈ ⁉ ⁎ ⁑ ⁂HTML UTF-8 SymbolsCurrency Symbols€ ₤ ₿ ₡ ₯ ₣ ₽Letterlike Symbols℃ ℉ Ω ⅂ ⅃ ⅄ ™Number FormsⅡ Ⅲ Ⅵ ⅛ ⅜ ⅝Math Operators∑ ∃ √ ∏ ≠ ∞ ⨎ ∅Math Symbols⟁ ⟀ ⟆ ⟮ ⟯ ⟨ ⟩ ⦀ ⧻ ⦿ ⧖Math Alphanumeric𝚷 𝚫 𝚽 𝛀 𝓪 𝓫 𝓬 𝓭Enclosed AlphanumericⒶ Ⓑ 🄰 🄱 🅐 🅑 ① ②Misc Technical⍇ ⍉ ⍾ ⍽ ⌬ ⌇Box Drawings┌ ┐ └ ┘ ├ ┤ ┬ ┴Block Elements▄ ▃ ▅ ▇ ░ ▒ ▓Geometric Shapes▣ ▧ ▲ △ ◐ ◒ ◧ ◪Weather Symbols☀ ☁ ☂ ☃ ★ ☆ ☇ ☈Astrological Symbols☿ ♁ ♃ ♆ ☼ ☽ ☾Musical Symbols𝄞 𝄆 𝄇 ♩ ♪ ♫ ♬Domino & Dices🁭🁵🂄 ⚁ ⚂ ⚄Mahjong Tiles🀀 🀁 🀃 🀅Chess Symbols♔ ♕ ♘ ♖Card Symbols♠ ♣ ♥ ♦ 🂡 🂣 🂨Arrows← ↑ → ↓ ⇐ ⇑ ⇒ ⇓Arrows A⟰ ⟱ ⟽ ⟾ ⟻ ⟼Arrows B⤀ ⤁ ⤺ ⤻ ⥀Arrows C🠈 🠉 🠊 🠋Symbols and Arrows⬮ ⬯ ⭅ ⭆Yin Yang Symbols☯ ☰ ☱ ☲ ☮ ☸ ☣Recycling Symbols♲ ♻ ♾ ☢Life & Religion⛨ ☮ ♰ ♀ ♂Places & Traffic⛱ ⛴ ⛟ ⛐ ⚠Misc Symbols☊ ⚖ ⛋ ⚜ ☘Dingbats✀ ✇ ✈ ✎ ✔ ✜ ✱Braille⠓⠑⠇⠇⠕ ⠺⠕⠗⠇⠙Aegean Numbers𐄊 𐄎 𐄗 𐄞 𐄧 𐄳Gothic𐌱 𐌸 𐌺 𐌼 𐌴 𐌶Alchemical🜉 🜊 🜋 🜬 🜑 🜎Meroitic Hieroglyphs𐦁 𐦕 𐦈 𐦆 𐦉 𐦀Egyptian Hieroglyphs𓀀 𓀄 𓀘 𓀢 𓀬Colored Symbols⌚ ⏩ ⏪ ☔ ⚽ ⚾HTML UTF-8 EmojisSmileys😀 😂 😊 😎 😜Hands✌ ✊ ☝ ✋ 👌People👮 🧕 👦 💏 🤴Office📈 💻 📌 📆 📒Places⛺ 🌋 🗽 🗿 🏢Transport🚈 🚗 🚢 🚌 🚀Animals🐴 🐕 🐘 🐻 🐞Food☕ 🌭 🍞 🍩 🍣Plants🌴 🌳 🌼 🍁 🥑Fruits🍇 🍊 🍏 🥝 🥥Sports⚽ 🏆 🤿 🏋 ⛳Earth & Sky🌐 🌍 🌖 🌟 🌞Weather⛅ ☔ 🌈 🌂 ⛄Clothing👚 👕 🎩 👜 👠Audio/Video🎥 🎵 🎹 🔊 📺Celebration🎁 🎃 🎈 🎓 🎂Entertainment🎨 🎪 🎭 🎡 🎢Symbols💡 💰 🔐 🔞 🔔"
tokens = text.encode("utf-8") 

# Bytes
tokens_Bytes = list(map(int, tokens))

print('Original')
print(text)
print("length:", len(text))
print('Bytes only (1-4B s / char)')
print(tokens_Bytes)
print("length:", len(tokens_Bytes))

Original
Basic LatinABCD abcd 0123 ?#$%Latin Extended A-BĀĂĄ ĆĈĊ ĒĔĖĘ Ⱡ Ɽ Ɱ ⱲLatin IPA Extentionsɖɜɣ ɘɫɛ ɱɷɞSpacing Modifierspʰ pʱ pʲ pʳDiacritical Marksàáâã èéêẽ òóôõGreek and CopticΑ Β Γ Δ Ε Ζ Η Θ Ι Κ ΛSlavic / CyrillicА Б В Г Д Е Ё Ж З И ЙGeorgianႠ Ⴁ Ⴂ Ⴃ Ⴄ Ⴅ Ⴆ Ⴇ Ⴈ Ⴉ ႪArmenianԱ Բ Գ Դ Ե Զ Է Ը Թ ԺGlagoliticⰀ Ⰱ Ⰲ Ⰳ Ⰴ Ⰵ Ⰶ Ⰷ Ⰸ ⰉArabicا ب ت ث ج ح خ د ذ ر زHebrewב ג ד ה ו ז ח ט י כ לSyriacܐ ܒ ܓ ܕ ܗ ܘ ܙ ܚ ܛ ܝ ܟHindi / Devanagariक ख ग घ ङ च छ ज झ ञ टThaiก ข ฃ ค ฅ ฆ ง จ ฉ ช ซEthiopicሀ ሁ ሂ ሃ ሄ ህ ሆ ሇ ለ ሉ ሊCherokeeᎠ Ꭱ Ꭲ Ꭳ Ꭴ Ꭵ Ꭶ Ꭷ Ꭸ Ꭹ ᎪAboriginalᐄ ᐅ ᐆ ᐇ ᐈ ᐉ ᐊ ᐋ ᐌ ᐍ ᐎRunicᚠ ᚢ ᚦ ᚨ ᚱ ᚲ ᚷ ᚹ ᚺ ᚾ ᛁGeneral Punctuation‰ ‱ ⁒ ‼ ⁇ ⁈ ⁉ ⁎ ⁑ ⁂HTML UTF-8 SymbolsCurrency Symbols€ ₤ ₿ ₡ ₯ ₣ ₽Letterlike Symbols℃ ℉ Ω ⅂ ⅃ ⅄ ™Number FormsⅡ Ⅲ Ⅵ ⅛ ⅜ ⅝Math Operators∑ ∃ √ ∏ ≠ ∞ ⨎ ∅Math Symbols⟁ ⟀ ⟆ ⟮ ⟯ ⟨ ⟩ ⦀ ⧻ ⦿ ⧖Math Alphanumeric𝚷 𝚫 𝚽 𝛀 𝓪 𝓫 𝓬 𝓭Enclosed AlphanumericⒶ Ⓑ 🄰 🄱 🅐 🅑 ① ②Misc Technical⍇ ⍉ ⍾ ⍽ ⌬ ⌇Box Drawings┌ ┐ └ ┘ ├ ┤ ┬ ┴Block Elements▄ ▃ ▅ ▇ ░ ▒ ▓Geometric Shapes▣ ▧ ▲ △ ◐ ◒ ◧ ◪Weather Symbols☀ ☁ ☂ ☃

In [3]:
def find_mode_pair(bytes):
    d = {}
    max_pair = (0,(-1,-1))
    for i,j in zip(bytes[:-1], bytes[1:]):
        keyy = str(i) + ';' + str(j)
        d[keyy] = d.get(keyy,0) + 1
        if max_pair[0] < d[keyy]:
            max_pair = (d[keyy] , (i,j))
    return max_pair, d
max_pair , counts= find_mode_pair(tokens_Bytes)
counts_sorted = sorted([ (v,k) for k,v in counts.items() ], reverse=True)
max_pair, counts_sorted 

((168, (32, 226)),
 [(168, '32;226'),
  (112, '32;240'),
  (103, '240;159'),
  (50, '32;225'),
  (27, '226;152'),
  (22, '115;226'),
  (21, '226;153'),
  (21, '105;99'),
  (20, '32;224'),
  (18, '240;144'),
  (18, '111;108'),
  (17, '109;98'),
  (16, '98;111'),
  (16, '83;121'),
  (16, '108;115'),
  (15, '32;83'),
  (15, '121;109'),
  (15, '101;114'),
  (14, '159;142'),
  (13, '97;110'),
  (13, '226;159'),
  (13, '131;32'),
  (13, '129;32'),
  (13, '105;110'),
  (12, '97;116'),
  (12, '159;140'),
  (12, '149;32'),
  (12, '136;32'),
  (12, '128;32'),
  (11, '240;157'),
  (11, '226;150'),
  (11, '225;144'),
  (11, '225;142'),
  (11, '225;136'),
  (11, '225;130'),
  (11, '224;184'),
  (11, '224;164'),
  (11, '138;32'),
  (11, '135;32'),
  (11, '134;32'),
  (11, '116;105'),
  (11, '115;240'),
  (10, '32;220'),
  (10, '32;216'),
  (10, '32;215'),
  (10, '32;208'),
  (10, '32;206'),
  (10, '226;176'),
  (10, '226;160'),
  (10, '226;156'),
  (10, '226;155'),
  (10, '225;154'),
  (10, '177;32'

In [4]:
i = 0
chr(int(counts_sorted[i][1].split(';')[0])) , chr(int(counts_sorted[i][1].split(';')[1]))

(' ', 'â')

In [5]:
def merge_and_replace(tokens, src, dest):
    res = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and tokens[i] == src[0] and tokens[i+1] == src[1]:
            res.append(dest)
            i += 1
        else:
            res.append(tokens[i])
        i += 1
    return res

Byte_Number = 255

new_token_bytes = merge_and_replace(tokens_Bytes, max_pair[1], Byte_Number + 1)
len(tokens_Bytes), len(new_token_bytes)

(2954, 2786)

In [6]:
def find_max_pair(bytes):
    d = {}
    max_pair = (0,(-1,-1))
    for i,j in zip(bytes[:-1], bytes[1:]):
        # keyy = str(i) + ';' + str(j)
        keyy = (i,j)
        d[keyy] = d.get(keyy,0) + 1
        if max_pair[0] < d[keyy]:
            max_pair = (d[keyy] , (i,j))
    return max_pair[1]

def merge_and_replace(tokens, src, dest):
    res = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and tokens[i] == src[0] and tokens[i+1] == src[1]:
            res.append(dest)
            i += 1
        else:
            res.append(tokens[i])
        i += 1
    return res

MAX_VOCAB_SIZE = 512

Byte_Number = 256
iters = MAX_VOCAB_SIZE - Byte_Number

def bpe(bytes, iters, byte_no_start = 256):

    merges_happened = {}
    bytes_curr = [*bytes]
    for i in range(iters):
        src = find_max_pair(bytes_curr)
        bytes_curr = merge_and_replace(bytes_curr, src, byte_no_start + i )
        merges_happened[src] = byte_no_start + i 
        
    return bytes_curr, merges_happened
    
new_tokens, merges = bpe(tokens_Bytes, iters, Byte_Number)
merges

{(32, 226): 256,
 (32, 240): 257,
 (257, 159): 258,
 (32, 225): 259,
 (115, 226): 260,
 (256, 152): 261,
 (105, 99): 262,
 (32, 224): 263,
 (240, 159): 264,
 (256, 153): 265,
 (111, 108): 266,
 (109, 98): 267,
 (83, 121): 268,
 (257, 144): 269,
 (101, 114): 270,
 (268, 267): 271,
 (271, 266): 272,
 (32, 272): 273,
 (97, 110): 274,
 (105, 110): 275,
 (97, 116): 276,
 (256, 159): 277,
 (273, 260): 278,
 (258, 142): 279,
 (32, 206): 280,
 (32, 208): 281,
 (259, 130): 282,
 (32, 216): 283,
 (32, 215): 284,
 (32, 220): 285,
 (263, 164): 286,
 (263, 184): 287,
 (259, 136): 288,
 (259, 142): 289,
 (259, 144): 290,
 (258, 140): 291,
 (32, 212): 292,
 (256, 176): 293,
 (259, 154): 294,
 (256, 150): 295,
 (257, 157): 296,
 (226, 160): 297,
 (256, 133): 298,
 (256, 154): 299,
 (256, 156): 300,
 (97, 108): 301,
 (256, 129): 302,
 (256, 148): 303,
 (65, 114): 304,
 (111, 114): 305,
 (101, 110): 306,
 (115, 264): 307,
 (256, 130): 308,
 (258, 141): 309,
 (256, 155): 310,
 (108, 101): 311,
 (105, 111

In [7]:
len(tokens_Bytes), len(new_tokens)

(2954, 1460)

In [8]:
print(f'Compression Ratio : {len(tokens_Bytes)/len(new_tokens)}')

Compression Ratio : 2.0232876712328767


In [9]:
def get_vocab(merges, Byte_Number = 256):
    vocab = { i:bytes([i]) for i in range(Byte_Number) }
    for (f1,f2), k in merges.items():
        vocab[k] = vocab[f1] + vocab[f2]
    return vocab
vocab = get_vocab(merges)    
len(vocab)

512

In [10]:
def decode(byte_tokens, vocab = vocab):
    return b"".join(vocab[bi] for bi in byte_tokens).decode('utf-8', errors='replace')
decode(tokens_Bytes,vocab)

'Basic LatinABCD abcd 0123 ?#$%Latin Extended A-BĀĂĄ ĆĈĊ ĒĔĖĘ Ⱡ Ɽ Ɱ ⱲLatin IPA Extentionsɖɜɣ ɘɫɛ ɱɷɞSpacing Modifierspʰ pʱ pʲ pʳDiacritical Marksàáâã èéêẽ òóôõGreek and CopticΑ Β Γ Δ Ε Ζ Η Θ Ι Κ ΛSlavic / CyrillicА Б В Г Д Е Ё Ж З И ЙGeorgianႠ Ⴁ Ⴂ Ⴃ Ⴄ Ⴅ Ⴆ Ⴇ Ⴈ Ⴉ ႪArmenianԱ Բ Գ Դ Ե Զ Է Ը Թ ԺGlagoliticⰀ Ⰱ Ⰲ Ⰳ Ⰴ Ⰵ Ⰶ Ⰷ Ⰸ ⰉArabicا ب ت ث ج ح خ د ذ ر زHebrewב ג ד ה ו ז ח ט י כ לSyriacܐ ܒ ܓ ܕ ܗ ܘ ܙ ܚ ܛ ܝ ܟHindi / Devanagariक ख ग घ ङ च छ ज झ ञ टThaiก ข ฃ ค ฅ ฆ ง จ ฉ ช ซEthiopicሀ ሁ ሂ ሃ ሄ ህ ሆ ሇ ለ ሉ ሊCherokeeᎠ Ꭱ Ꭲ Ꭳ Ꭴ Ꭵ Ꭶ Ꭷ Ꭸ Ꭹ ᎪAboriginalᐄ ᐅ ᐆ ᐇ ᐈ ᐉ ᐊ ᐋ ᐌ ᐍ ᐎRunicᚠ ᚢ ᚦ ᚨ ᚱ ᚲ ᚷ ᚹ ᚺ ᚾ ᛁGeneral Punctuation‰ ‱ ⁒ ‼ ⁇ ⁈ ⁉ ⁎ ⁑ ⁂HTML UTF-8 SymbolsCurrency Symbols€ ₤ ₿ ₡ ₯ ₣ ₽Letterlike Symbols℃ ℉ Ω ⅂ ⅃ ⅄ ™Number FormsⅡ Ⅲ Ⅵ ⅛ ⅜ ⅝Math Operators∑ ∃ √ ∏ ≠ ∞ ⨎ ∅Math Symbols⟁ ⟀ ⟆ ⟮ ⟯ ⟨ ⟩ ⦀ ⧻ ⦿ ⧖Math Alphanumeric𝚷 𝚫 𝚽 𝛀 𝓪 𝓫 𝓬 𝓭Enclosed AlphanumericⒶ Ⓑ 🄰 🄱 🅐 🅑 ① ②Misc Technical⍇ ⍉ ⍾ ⍽ ⌬ ⌇Box Drawings┌ ┐ └ ┘ ├ ┤ ┬ ┴Block Elements▄ ▃ ▅ ▇ ░ ▒ ▓Geometric Shapes▣ ▧ ▲ △ ◐ ◒ ◧ ◪Weather Symbols☀ ☁ ☂ ☃ ★ ☆ ☇ ☈

In [11]:
    
def encode(string_obj, merges = merges):
    bytes_obj = list(string_obj.encode('utf-8'))
    m = sorted([(v,k) for k,v in merges.items()])
    for k,v in m:
        bytes_obj = merge_and_replace(bytes_obj, v, k)
    return bytes_obj
tok_bytes = encode(text)
len(tok_bytes)

1460

In [12]:
text
text_2 = decode(encode(text), vocab)
text_2

'Basic LatinABCD abcd 0123 ?#$%Latin Extended A-BĀĂĄ ĆĈĊ ĒĔĖĘ Ⱡ Ɽ Ɱ ⱲLatin IPA Extentionsɖɜɣ ɘɫɛ ɱɷɞSpacing Modifierspʰ pʱ pʲ pʳDiacritical Marksàáâã èéêẽ òóôõGreek and CopticΑ Β Γ Δ Ε Ζ Η Θ Ι Κ ΛSlavic / CyrillicА Б В Г Д Е Ё Ж З И ЙGeorgianႠ Ⴁ Ⴂ Ⴃ Ⴄ Ⴅ Ⴆ Ⴇ Ⴈ Ⴉ ႪArmenianԱ Բ Գ Դ Ե Զ Է Ը Թ ԺGlagoliticⰀ Ⰱ Ⰲ Ⰳ Ⰴ Ⰵ Ⰶ Ⰷ Ⰸ ⰉArabicا ب ت ث ج ح خ د ذ ر زHebrewב ג ד ה ו ז ח ט י כ לSyriacܐ ܒ ܓ ܕ ܗ ܘ ܙ ܚ ܛ ܝ ܟHindi / Devanagariक ख ग घ ङ च छ ज झ ञ टThaiก ข ฃ ค ฅ ฆ ง จ ฉ ช ซEthiopicሀ ሁ ሂ ሃ ሄ ህ ሆ ሇ ለ ሉ ሊCherokeeᎠ Ꭱ Ꭲ Ꭳ Ꭴ Ꭵ Ꭶ Ꭷ Ꭸ Ꭹ ᎪAboriginalᐄ ᐅ ᐆ ᐇ ᐈ ᐉ ᐊ ᐋ ᐌ ᐍ ᐎRunicᚠ ᚢ ᚦ ᚨ ᚱ ᚲ ᚷ ᚹ ᚺ ᚾ ᛁGeneral Punctuation‰ ‱ ⁒ ‼ ⁇ ⁈ ⁉ ⁎ ⁑ ⁂HTML UTF-8 SymbolsCurrency Symbols€ ₤ ₿ ₡ ₯ ₣ ₽Letterlike Symbols℃ ℉ Ω ⅂ ⅃ ⅄ ™Number FormsⅡ Ⅲ Ⅵ ⅛ ⅜ ⅝Math Operators∑ ∃ √ ∏ ≠ ∞ ⨎ ∅Math Symbols⟁ ⟀ ⟆ ⟮ ⟯ ⟨ ⟩ ⦀ ⧻ ⦿ ⧖Math Alphanumeric𝚷 𝚫 𝚽 𝛀 𝓪 𝓫 𝓬 𝓭Enclosed AlphanumericⒶ Ⓑ 🄰 🄱 🅐 🅑 ① ②Misc Technical⍇ ⍉ ⍾ ⍽ ⌬ ⌇Box Drawings┌ ┐ └ ┘ ├ ┤ ┬ ┴Block Elements▄ ▃ ▅ ▇ ░ ▒ ▓Geometric Shapes▣ ▧ ▲ △ ◐ ◒ ◧ ◪Weather Symbols☀ ☁ ☂ ☃ ★ ☆ ☇ ☈

'Basic LatinABCD abcd 0123 ?#$%Latin Extended A-BĀĂĄ ĆĈĊ ĒĔĖĘ Ⱡ Ɽ Ɱ ⱲLatin IPA Extentionsɖɜɣ ɘɫɛ ɱɷɞSpacing Modifierspʰ pʱ pʲ pʳDiacritical Marksàáâã èéêẽ òóôõGreek and CopticΑ Β Γ Δ Ε Ζ Η Θ Ι Κ ΛSlavic / CyrillicА Б В Г Д Е Ё Ж З И ЙGeorgianႠ Ⴁ Ⴂ Ⴃ Ⴄ Ⴅ Ⴆ Ⴇ Ⴈ Ⴉ ႪArmenianԱ Բ Գ Դ Ե Զ Է Ը Թ ԺGlagoliticⰀ Ⰱ Ⰲ Ⰳ Ⰴ Ⰵ Ⰶ Ⰷ Ⰸ ⰉArabicا ب ت ث ج ح خ د ذ ر زHebrewב ג ד ה ו ז ח ט י כ לSyriacܐ ܒ ܓ ܕ ܗ ܘ ܙ ܚ ܛ ܝ ܟHindi / Devanagariक ख ग घ ङ च छ ज झ ञ टThaiก ข ฃ ค ฅ ฆ ง จ ฉ ช ซEthiopicሀ ሁ ሂ ሃ ሄ ህ ሆ ሇ ለ ሉ ሊCherokeeᎠ Ꭱ Ꭲ Ꭳ Ꭴ Ꭵ Ꭶ Ꭷ Ꭸ Ꭹ ᎪAboriginalᐄ ᐅ ᐆ ᐇ ᐈ ᐉ ᐊ ᐋ ᐌ ᐍ ᐎRunicᚠ ᚢ ᚦ ᚨ ᚱ ᚲ ᚷ ᚹ ᚺ ᚾ ᛁGeneral Punctuation‰ ‱ ⁒ ‼ ⁇ ⁈ ⁉ ⁎ ⁑ ⁂HTML UTF-8 SymbolsCurrency Symbols€ ₤ ₿ ₡ ₯ ₣ ₽Letterlike Symbols℃ ℉ Ω ⅂ ⅃ ⅄ ™Number FormsⅡ Ⅲ Ⅵ ⅛ ⅜ ⅝Math Operators∑ ∃ √ ∏ ≠ ∞ ⨎ ∅Math Symbols⟁ ⟀ ⟆ ⟮ ⟯ ⟨ ⟩ ⦀ ⧻ ⦿ ⧖Math Alphanumeric𝚷 𝚫 𝚽 𝛀 𝓪 𝓫 𝓬 𝓭Enclosed AlphanumericⒶ Ⓑ 🄰 🄱 🅐 🅑 ① ②Misc Technical⍇ ⍉ ⍾ ⍽ ⌬ ⌇Box Drawings┌ ┐ └ ┘ ├ ┤ ┬ ┴Block Elements▄ ▃ ▅ ▇ ░ ▒ ▓Geometric Shapes▣ ▧ ▲ △ ◐ ◒ ◧ ◪Weather Symbols☀ ☁ ☂ ☃ ★ ☆ ☇ ☈

In [13]:
text_2 == text

True

In [14]:
ttext = "ng https://download.pytorch.org/whl/triton-3.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (166.7 MB)━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.7/166.7 MB 18.5 MB/s  0:00:09Downloading https://download.pytorch.org/whl/pillow-11.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (4.4 MB)━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 2.3 MB/s  0:"
ttext_2 = decode(encode(ttext), vocab)
ttext_2

'ng https://download.pytorch.org/whl/triton-3.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (166.7 MB)━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.7/166.7 MB 18.5 MB/s  0:00:09Downloading https://download.pytorch.org/whl/pillow-11.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (4.4 MB)━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 2.3 MB/s  0:'

In [15]:
ttext == ttext_2

True

### Module - implementation

In [ ]:
from bpe.bpe_plain import myTokenizer

bpe_plain = myTokenizer()

In [ ]:
with open(f'{data_dir}virat_wiki.txt') as f:
    sample_text = f.read()
sample_text[:200]    

'Virat Kohli (Hindi pronunciation: [ʋɪˈɾaːʈᵊ ˈkoːɦᵊliː] ⓘ, born 5 November 1988) is an Indian international cricketer and the former captain of the Indian national cricket team. He is a right-handed ba'

In [18]:
bpe_plain.train(sample_text, 10000) 

Training Progress: 100%|██████████| 9744/9744 [00:35<00:00, 278.35merges/s]


In [19]:
zibb = bpe_plain.encode(sample_text)
zibb[:10]

[9999, 3561, 707, 367, 1980, 1143, 272, 1976, 1521, 1416]

In [20]:
outtext = bpe_plain.decode(zibb)
outtext[:100]

'Virat Kohli (Hindi pronunciation: [ʋɪˈɾaːʈᵊ ˈkoːɦᵊliː] ⓘ, born 5 November 1988) is an Indian interna'

In [21]:
outtext == sample_text

True

In [ ]:
bpe_plain.save(f'{models_dir}virat_wiki_bpe')

In [ ]:
bpe_plain.load(f'{models_dir}virat_wiki_bpe.model')

In [ ]:
with open(f'{data_dir}rohit_wiki.txt') as f:
    rohit_sample_text = f.read()
rohit_sample_text[:200]

'Rohit Gurunath Sharma (born 30 April 1987) is an Indian international cricketer and the captain of the India national team in ODIs. He is also a former captain in Tests and T20Is. He is widely regarde'

In [25]:
rohit_zibb =bpe_plain.encode(rohit_sample_text)
rohit_outtext = bpe_plain.decode(zibb)
rohit_outtext == rohit_sample_text

False

### Issues

The BPE is basic and can have several practical tokenization issues related to handling whitespace, punctuation, and subword units. 

These create inconsistencies and reduce quality in downstream tasks. Few problems include:

1. **Whitespace Handling**
   - Spaces are treated as normal tokens and may not merge consistently or meaningfully with words. (like python indent code issue)
   - Leading, trailing, or multiple spaces can create fragmented tokens that don't merge well

2. **Punctuation Issues**
   - Punctuation marks (e.g., commas, periods, exclamations, quotes) are often not tokenized separately or properly merged
   - Different usages of punctuation with the same word create multiple tokens for similar forms (e.g., "word" vs "word!" as in GPT-2 paper)

3. **Apostrophes and Contractions**
   - Apostrophes (') in contractions or possessives can create token splits with no semantic meaning (e.g., "don't", "John's").

4. **Out-of-Vocabulary or Rare Tokens**
   - Rare tokens or typos may not merge well and can remain fragmented (Need to have some kind of fallback representation)

5. **PLacement of the word**
   - placement like starting word and middle of sentence word might get different tokenizations, which should be avoided decreasing redundancy



### Required optimization - solved by GPT-2, LLaMA-2 (tiktoken, Sentencepiece)

solution
- split the word with a space in front of it
- dont merge numbers and word

[GPT-2](https://github.com/openai/gpt-2/blob/master/src/encoder.py) uses this regex: `r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""`



GPT-4 uses this regex: `r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""`

---
**Conclusion**: better to use the sentencepiece for training custom tokenization